# OptimEngine Showcase

This notebook demonstrates the low-level optimization pieces directly. It intentionally uses direct construction only: the point is to show how `PositionSplit`, `Batches`, `NegLogProbLoss`, `Optimizer`, `Stopper`, and `OptimEngine` fit together.

In [1]:
import jax
import jax.numpy as jnp
import optax
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl
import liesel.optim as opt


def normal_model(n=32, *, seed=1, name="y"):
    key = jax.random.key(seed)
    y_values = 1.4 + 0.35 * jax.random.normal(key, (n,))
    loc = lsl.Var.new_param(jnp.array(0.0), name="loc")
    y = lsl.Var.new_obs(
        y_values,
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name=name,
    )
    return lsl.Model([y])


def two_branch_model(*, seed=10):
    key1, key2 = jax.random.split(jax.random.key(seed))
    loc = lsl.Var.new_param(jnp.array(0.0), name="loc")
    y1_values = 1.2 + 0.25 * jax.random.normal(key1, (30,))
    y2_values = -0.6 + 0.40 * jax.random.normal(key2, (11,))
    y1 = lsl.Var.new_obs(
        y1_values,
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name="y_long",
    )
    y2 = lsl.Var.new_obs(
        y2_values,
        lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name="y_short",
    )
    return lsl.Model([y1, y2])

## Build a Small Model

The target is a normal likelihood with one unknown location parameter.

In [2]:
model = normal_model(seed=1)

{
    "parameters": list(model.parameters),
    "observed": list(model.observed),
    "initial_loc": float(model.extract_position(["loc"])["loc"]),
}

{'parameters': ['loc'], 'observed': ['y'], 'initial_loc': 0.0}

## Assemble The Engine Components

A direct engine needs a split, batches, a loss, optimizers, a stopper, and an initial model state.

In [ ]:
split = opt.PositionSplit.from_model(
    model,
    validate_axis_share=0.2,
    shuffle=True,
    seed=11,
)
batches = opt.Batches(
    ["y"], axis_size=split.train_axis_size, batch_size=None, shuffle=False
)
loss = opt.NegLogProbLoss(model, split, scale=True)
optimizer = opt.Optimizer(
    list(model.parameters), optax.adam(0.05), identifier="loc_adam"
)
stopper = opt.Stopper(epochs=6, patience=6)

{
    "split_sizes": {
        "train": split.train_axis_size,
        "validate": split.validate_axis_size,
        "test": split.test_axis_size,
    },
    "batch_size": batches.batch_size,
    "loss_scaled": loss.scale,
    "optimizer_keys": optimizer.position_keys,
    "stopper_epochs": stopper.epochs,
}

## Construct And Inspect An `OptimEngine`

The engine is the low-level loop object. During fitting, it reports progress by default.

In [4]:
engine = opt.OptimEngine(
    loss=loss,
    optimizers=[optimizer],
    stopper=stopper,
    batches=batches,
    initial_state=model.state,
    seed=123,
    save_position_history=True,
)

{
    "position_keys": engine.position_keys,
    "batch_keys": engine.batches.position_keys,
    "has_validation": engine.split.has_validation,
    "train_monitor": engine.train_monitor,
}

{'position_keys': ['loc'],
 'batch_keys': ['y'],
 'has_validation': True,
 'train_monitor': 'auto'}

## Fit And Inspect The Result

The result stores final and best positions plus a compact optimization history.

In [5]:
result = engine.fit()

{
    "final_epoch": result.final_epoch,
    "best_epoch": result.best_epoch,
    "best_loc": round(float(result.best_position["loc"]), 4),
    "loss_head": result.history.loss_df().head().round(4).to_dict("records"),
}

Initializing:   0%|          | 0/6 [00:00<?, ?it/s]

Initializing:  17%|█▋        | 1/6 [00:00<00:01,  4.87it/s]

Training loss: 1.813, Monitoring loss: 2.258:  17%|█▋        | 1/6 [00:00<00:01,  4.87it/s]

Training loss: 1.749, Monitoring loss: 2.179:  33%|███▎      | 2/6 [00:00<00:00,  4.87it/s]

Training loss: 1.688, Monitoring loss: 2.102:  50%|█████     | 3/6 [00:00<00:00,  4.87it/s]

...

Training loss: 1.574, Monitoring loss: 1.957:  83%|████████▎ | 5/6 [00:00<00:00,  4.87it/s]

Training loss: 1.522, Monitoring loss: 1.888: 100%|██████████| 6/6 [00:00<00:00,  4.87it/s]

Training loss: 1.522, Monitoring loss: 1.888: 100%|██████████| 6/6 [00:00<00:00,  4.87it/s]

Training loss: 1.522, Monitoring loss: 1.888: 100%|██████████| 6/6 [00:00<00:00, 25.16it/s]

{'final_epoch': 6,
 'best_epoch': 5,
 'best_loc': 0.2982,
 'loss_head': [{'epoch': 0.0, 'loss_train': 1.8129, 'loss_validate': 2.2582},
  {'epoch': 1.0, 'loss_train': 1.7494, 'loss_validate': 2.1788},
  {'epoch': 2.0, 'loss_train': 1.6884, 'loss_validate': 2.1021},
  {'epoch': 3.0, 'loss_train': 1.6301, 'loss_validate': 2.0281},
  {'epoch': 4.0, 'loss_train': 1.5745, 'loss_validate': 1.9568}]}

## Mini-Batches And Validation

Switching from full-data training to mini-batch training only changes the `Batches` object and, here, the train monitor.

In [ ]:
mini_batches = opt.Batches(
    ["y"], axis_size=split.train_axis_size, batch_size=8, shuffle=True
)
mini_engine = opt.OptimEngine(
    loss=loss,
    optimizers=[
        opt.Optimizer(list(model.parameters), optax.adam(0.03), identifier="mini_adam")
    ],
    stopper=opt.Stopper(epochs=4, patience=4),
    batches=mini_batches,
    initial_state=model.state,
    seed=456,
    train_monitor="weighted_epoch_average",
)
mini_result = mini_engine.fit()

{
    "n_full_batches": mini_engine.batches.n_full_batches,
    "batch_size": mini_engine.batches.batch_size,
    "train_monitor": mini_engine.train_monitor,
    "final_epoch": mini_result.final_epoch,
    "best_loc": round(float(mini_result.best_position["loc"]), 4),
}

## Multi-Size Branches With Managers

For models with observed branches of different lengths, use `PositionSplitManager` and `BatchManager` explicitly with the direct engine.

In [ ]:
multi_model = two_branch_model(seed=20)
multi_split = opt.PositionSplit.from_model(
    multi_model,
    validate_axis_share=0.2,
    shuffle=True,
    seed=21,
    multi_size="manager",
)
children = [
    opt.Batches(
        child.position_keys,
        axis_size=child.train_axis_size,
        batch_size=min(6, child.train_axis_size),
        shuffle=True,
    )
    for child in multi_split.splits
]
multi_batches = opt.BatchManager(children, mode="resample", epoch_size="max")
multi_loss = opt.NegLogProbLoss(multi_model, multi_split, scale=True)
multi_engine = opt.OptimEngine(
    loss=multi_loss,
    optimizers=[
        opt.Optimizer(
            list(multi_model.parameters), optax.adam(0.03), identifier="multi_adam"
        )
    ],
    stopper=opt.Stopper(epochs=4, patience=4),
    batches=multi_batches,
    initial_state=multi_model.state,
    seed=789,
)
multi_result = multi_engine.fit()

{
    "split_type": type(multi_engine.split).__name__,
    "batch_type": type(multi_engine.batches).__name__,
    "train_axis_sizes": multi_engine.split.train_axis_sizes,
    "n_full_batches": multi_engine.batches.n_full_batches,
    "best_loc": round(float(multi_result.best_position["loc"]), 4),
}